# Project: Analysis of NYPD Arrest Data (Year to Date)

## 1. Introduction & Research Question
**Dataset:** NYPD Arrest Data (Year to Date). This dataset includes every arrest effected in NYC by the NYPD during the current year.
**Source:** NYC Open Data API.

**Research Question:** Do arrest demographics (specifically Race) shift significantly between low-level offenses (Misdemeanors) and severe crimes (Felonies)? 
*Hypothesis:* We might observe different racial distributions depending on the severity of the alleged crime.

**Methodology:**
1. Load data via Socrata API (fetching 50,000 rows to ensure statistical significance).
2. Clean and map the `LAW_CAT_CD` column (Level of Offense).
3. Aggregate data by Offense Level and Race.
4. Visualize the proportional distribution using a Stacked Bar Chart.

In [28]:
import requests
import pandas as pd
import plotly.io as pio
import plotly.express as px

In [29]:
response=requests.get("https://data.cityofnewyork.us/resource/uip8-fykc.json")
data=response.json()
data

df=pd.DataFrame(data)
print(f"Data Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")

Data Loaded Successfully: 1000 rows, 24 columns


## 2. Data Inspection
According to the Data Dictionary, the key columns for our research are:
* `LAW_CAT_CD`: Level of offense (F=Felony, M=Misdemeanor, V=Violation).
* `PERP_RACE`: Race of the suspect.
* `ARREST_KEY`: Unique identifier for the arrest.

Let's inspect the first few rows and the unique values for the offense level.

In [30]:
# Inspect the first 5 rows
display(df.head())

# Check unique values in the 'Level of Offense' column to understand the codes
# Based on Data Dictionary: F=Felony, M=Misdemeanor, V=Violation
print("Unique Offense Levels found:", df['law_cat_cd'].unique())

# Check for missing values in critical columns
print("\nMissing values check:")
print(df[['law_cat_cd', 'perp_race']].isnull().sum())

,arrest_key,arrest_date,pd_cd,pd_desc,ky_cd,ofns_desc,law_code,law_cat_cd,arrest_boro,arrest_precinct,...,x_coord_cd,y_coord_cd,latitude,longitude,geocoded_column,:@computed_region_f5dn_yrer,:@computed_region_yeji_bk3q,:@computed_region_92fq_4b7q,:@computed_region_sbqj_enih,:@computed_region_efsh_h5xi
0,298760433,2025-01-02T00:00:00.000,782,"WEAPONS, POSSESSION, ETC",236,DANGEROUS WEAPONS,PL 2650101,M,Q,115,...,0,0,0.0,0.0,"{'type': 'Point', 'coordinates': [0, 0]}",NaN,NaN,NaN,NaN,NaN
1,299030225,2025-01-07T00:00:00.000,105,STRANGULATION 1ST,106,FELONY ASSAULT,PL 1211200,F,M,28,...,997439,233857,40.808558,-73.952357,"{'type': 'Point', 'coordinates': [-73.952357, ...",18,4,36,18,12424
2,299127494,2025-01-08T00:00:00.000,849,"NY STATE LAWS,UNCLASSIFIED VIO",677,OTHER STATE LAWS,LOC00000V0,V,K,81,...,0,0,0.0,0.0,"{'type': 'Point', 'coordinates': [0, 0]}",NaN,NaN,NaN,NaN,NaN
3,299188536,2025-01-09T00:00:00.000,259,"CRIMINAL MISCHIEF,UNCLASSIFIED 4",351,CRIMINAL MISCHIEF & RELATED OF,PL 1450001,M,M,7,...,0,0,0.0,0.0,"{'type': 'Point', 'coordinates': [0, 0]}",NaN,NaN,NaN,NaN,NaN
4,299533742,2025-01-16T00:00:00.000,155,RAPE 2,104,RAPE,PL 1303001,F,K,81,...,1005319,190473,40.6894642952604,-73.9240290899499,"{'type': 'Point', 'coordinates': [-73.92402908...",69,2,17,52,18181


Unique Offense Levels found: ['M' 'F' 'V' nan '9' 'I']

Missing values check:
law_cat_cd    2
perp_race     0
dtype: int64


## 3. Data Cleaning & Preparation
To ensure accurate visualization:
1.  We will drop rows where `law_cat_cd` is missing.
2.  We will map the single-letter codes ('F', 'M', 'V') to their full names ('Felony', 'Misdemeanor', 'Violation') for better readability in the charts.

In [31]:
# 1. Drop rows with missing Level of Offense
df_clean = df.dropna(subset=['law_cat_cd']).copy()

# 2. Map the codes to full names
# Dictionary mapping based on NYPD metadata
level_map = {
    'F': 'Felony (Severe)',
    'M': 'Misdemeanor (Minor)',
    'V': 'Violation (Cite)',
    'I': 'Infraction' # Creating a catch-all just in case
}

df_clean['offense_level'] = df_clean['law_cat_cd'].map(level_map)

# Check the distribution of cleaned data
print(df_clean['offense_level'].value_counts())

offense_level
Misdemeanor (Minor)    597
Felony (Severe)        388
Violation (Cite)         9
Infraction               1
Name: count, dtype: int64


## 4. Analysis & Visualization
**Goal:** We want to compare the *percentage* composition of each race within each crime category.
**Visualization Plan:** A **100% Stacked Bar Chart**. 
* **X-Axis:** Offense Level (Felony vs. Misdemeanor Vs. Violation Vs.Infraction ).
* **Y-Axis:** Proportion of Arrests (0-100%).
* **Color:** Suspect Race.

This visualization will allow us to see if certain racial groups are disproportionately represented in specific types of arrests (e.g., lower-level offenses) relative to others.

I chose Plotly Express for this visualization because it handles categorical data aggregations efficiently. Unlike static charts, this interactive plot allows readers to hover over specific segments to see the exact counts and percentages for each racial group, making the comparison between Felonies and Misdemeanors more precise.

In [32]:
# 1. Data Preparation for Plotly

df_counts = df_clean.groupby(['offense_level', 'perp_race']).size().reset_index(name='counts')

# 2. Calculate Percentage
df_counts['percentage'] = df_counts.groupby('offense_level')['counts'].transform(lambda x: x / x.sum())

# 3. Create the Interactive Plot
fig = px.bar(
    df_counts, 
    x='offense_level', 
    y='percentage', 
    color='perp_race',
    title='Racial Distribution of Arrests by Offense Severity (Interactive)',
    labels={
        'offense_level': 'Level of Offense', 
        'percentage': 'Proportion', 
        'perp_race': 'Suspect Race'
    },
    text_auto='.1%', 
    template='plotly_white' 
)

# 4. Fine-tuning the layout
fig.update_layout(
    yaxis_tickformat=".0%",
    legend_title_text='Suspect Race'
)


fig.show()

## 5. Conclusion & Observations
Based on the visualization above:

**Observation 1(Overall Disparity)**: The visualization reveals a stark racial disparity in arrest numbers. We observe that for all the offence types recorded, Asian and White arrests represent a significantly tiny portion of the total.Black and White Hispanic suspects combined appear to constitue the vast majority of all arrests, regardless of the the offense leve

**Observation 2**: Interestingly, the racial composition does not shift dramatically when moving from **Misdemeanors** to **Felonies**. This indicates that the racial disparities in policing are structural and persist regardless of whether the alleged crime is a minor infraction or a serious felony.

**Conclusion:**
This analysis answers the research question by showing that arrest demographics are relatively stable across offense levels, but highly skewed racially. This suggests that factors beyond just "crime severity" likely influence arrest patterns in New York City.


